# Graphlib, Codecs And Misc Modules

## Quick notes

### What this covers
`graphlib` (dependency ordering), `codecs` (text encodings in depth), and a quick tour of small standard-library modules: `getpass`, `mimetypes`, `colorsys`, `shlex`, `fileinput`, `tokenize`/`keyword`, `zipapp`, `pkgutil`/`importlib.resources`, `platform`/`sysconfig`, `site`/`sys.path`. `webbrowser`, `smtplib`, `tkinter` and `turtle` are explained in comments only because they cause side effects or need a display.

### Key points
- **`graphlib.TopologicalSorter({node: {deps}})`**: `static_order()` gives a valid "dependencies first" order. Several orders can be valid; rely only on the rule.
- **Parallel waves**: `prepare()`, then loop `while ts.is_active(): ready = ts.get_ready(); ...; ts.done(*ready)`. Everything in one `get_ready()` batch can run at the same time.
- **`CycleError`** (a `ValueError`) when nodes depend on each other; `err.args[1]` is one cycle. Nodes must be hashable.
- **`codecs`**: `encode/decode`, `lookup(name).name` normalizes names, bytes-to-bytes codecs exist (`hex`, `zlib`, `base64`).
- **Error handlers** (`errors=`): `strict`, `ignore`, `replace`, `backslashreplace`, `xmlcharrefreplace`, `namereplace`, `surrogateescape`; add your own with `codecs.register_error`.
- **Incremental decoders** (`getincrementaldecoder`, `iterdecode`) keep half-received multi-byte characters between chunks; pass `final=True` at the end.
- **`open(path, encoding=...)`** is the modern tool; `codecs.open` is legacy (binary underneath, no newline translation, splits on extra separators like U+2028).
- **Custom codec**: `codecs.register(search_function)` returning a `CodecInfo`.
- **BOM**: `utf-8-sig` writes/strips the UTF-8 BOM; plain `utf-16` writes one; `utf-16-le/be` never do.
- **`getpass.getpass`** reads without echo (not run here); `getuser()` is for labels, not security.
- **`mimetypes.guess_type`** works on names only; **`colorsys`** uses floats 0.0 to 1.0.
- **`shlex.split/quote/join`** follow shell quoting; `str.split` breaks quoted arguments.
- **`fileinput`** treats many files as one stream, with `inplace=True` to edit files (print goes into the file).
- **`tokenize`** keeps comments (unlike `ast`); `keyword.kwlist` / `softkwlist`.
- **`zipapp.create_archive`** builds a runnable `.pyz` (pure Python only).
- **`importlib.resources.files(pkg)`** reads package data safely (works in zips); `pkgutil.iter_modules` lists modules without importing.
- **`platform`, `sysconfig`, `sys.platform`** describe the machine and Python build; **`sys.path`** order decides which module wins.

### Gotchas (the part worth revising twice)
- `errors="ignore"` silently loses data; `replace` hides problems. Choose deliberately, and prefer `strict` or `surrogateescape` when you must round-trip.
- Decoding network chunks one by one with `bytes.decode` fails on split characters; use an incremental decoder.
- Excel CSVs often start with a BOM that appears as `﻿` in the first header; read with `utf-8-sig`.
- Only one `fileinput.input()` can be active at once; `inplace=True` really rewrites files (use `backup=".bak"` when unsure).
- `getpass` falls back to echoing when there is no terminal. It does not protect secrets in memory.
- `zipapp` cannot load C extensions from inside the zip.
- Do not build data paths from `__file__`; use `importlib.resources`.
- A local file named `random.py` or `json.py` shadows the standard module (`sys.path[0]` is searched first).
- `static_order()` raises `CycleError` only when the iteration runs, not at construction; call `prepare()` to detect it early.
- `tkinter`/`turtle` block in `mainloop()`, fail without a display (`TclError`) and some Python builds lack Tk.
- Never call `webbrowser.open` or `smtplib` in tests or demos; mock them. Build emails with `EmailMessage`, not string concatenation (header injection).
- Do not parse `platform` strings or print `node()` into shared logs.

### When it matters
Build/task runners and installers (`graphlib`), any code reading or writing files in unknown encodings or streaming bytes (`codecs`), command-line tooling (`shlex`, `fileinput`), packaging and plugin loading (`zipapp`, `pkgutil`, `importlib.resources`), and portability checks (`platform`, `sysconfig`).

### One-line recap example
```python
import graphlib
print(list(graphlib.TopologicalSorter({"app": {"lib"}, "lib": {"core"}}).static_order()))
# ['core', 'lib', 'app']
```

In [1]:
# notebooks have no __file__; point it at this notebook
__file__ = str(__import__("pathlib").Path("24-Graphlib-Codecs-And-Misc-Modules.ipynb").resolve())

In [2]:
# notebook shim: ipykernel re-installs its own input() before every cell, so patch
# the kernel's raw_input instead; then input() reads sys.stdin like a script does.
import sys
def _input(prompt=""):
    sys.stdout.write(prompt); sys.stdout.flush()
    line = sys.stdin.readline()
    if not line:
        raise EOFError("EOF when reading a line")
    return line.rstrip("\n")
get_ipython().kernel.raw_input = _input

In [3]:
import codecs
import colorsys
import fileinput
import getpass
import graphlib
import importlib
import importlib.resources
import importlib.util
import io
import keyword
import mimetypes
import os
import pkgutil
import platform
import shlex
import site
import sys
import sysconfig
import tempfile
import tokenize
import zipapp

## 01. Index - Graphs, Text Encodings And A Tour Of Small Modules

- 01 Index                         13 fileinput
- 02 graphlib basics               14 tokenize and keyword
- 03 prepare/get_ready/done        15 zipapp
- 04 CycleError                    16 pkgutil and importlib.resources
- 05 codecs: encode/decode         17 platform and sysconfig
- 06 error handlers                18 site and sys.path recap
- 07 incremental decoders          19 webbrowser and smtplib (comment only)
- 08 codecs.open vs open           20 tkinter and turtle (comment only)
- 09 custom codec + BOM            21 What to remember
- 10 getpass (and getuser)
- 11 mimetypes and colorsys
- 12 shlex
- Each module here is small, but each one removes a chunk of hand-written
  code. Nothing below opens a browser, sends mail, prompts, or leaves files.

## 02. graphlib.TopologicalSorter - Ordering With Dependencies

- A topological order lists tasks so that every task comes AFTER the tasks it
  depends on. Think: build steps, course prerequisites, install order.
- TopologicalSorter takes a graph as {node: iterable_of_predecessors}:
  "node needs all of these first".
- static_order() returns the whole order at once (an iterator).
- Several valid orders can exist; the library picks one. Do not rely on
  which, only on the "dependencies first" rule.

In [4]:
graph = {"app": {"lib", "config"}, "lib": {"core"}, "config": {"core"}, "core": set()}
ts = graphlib.TopologicalSorter(graph)
order = list(ts.static_order())
print("order:", order)
print("core first:", order[0] == "core", "| app last:", order[-1] == "app")
print("each node after its dependencies:", all(order.index(d) < order.index(n) for n, deps in graph.items() for d in deps))
ts2 = graphlib.TopologicalSorter()
ts2.add("c", "a", "b")  # c depends on a and b
ts2.add("b", "a")
print("built with add():", list(ts2.static_order()))

order: ['core', 'lib', 'config', 'app']
core first: True | app last: True
each node after its dependencies: True
built with add(): ['a', 'b', 'c']


## 03. prepare / get_ready / done - Running Tasks In Parallel Waves

- static_order() is fine for a plain loop. For PARALLEL work use the manual API:
    prepare()    freeze the graph (and detect cycles)
    get_ready()  tuple of nodes whose dependencies are all finished
    done(*nodes) tell the sorter these are finished; this unlocks the next ones
    is_active()  True while there is still work to hand out or finish
- Everything returned by get_ready() together can run at the same time.

In [5]:
ts3 = graphlib.TopologicalSorter({"d": {"b", "c"}, "b": {"a"}, "c": {"a"}})
ts3.prepare()
wave = 0
while ts3.is_active():
    ready = ts3.get_ready()
    wave += 1
    print(f"wave {wave}:", sorted(ready))
    ts3.done(*ready)
print("waves needed (the longest chain):", wave)
try:
    ts3.add("e", "d")
except ValueError as err:
    print("cannot add after prepare():", err)

wave 1: ['a']
wave 2: ['b', 'c']
wave 3: ['d']
waves needed (the longest chain): 3
cannot add after prepare(): Nodes cannot be added after a call to prepare()


## 04. CycleError - Circular Dependencies

- If A needs B and B needs A, no order exists. prepare() / static_order()
  raise graphlib.CycleError (a ValueError subclass).
- err.args[1] holds one cycle as a list of nodes - useful in the message.

In [6]:
try:
    list(graphlib.TopologicalSorter({"a": {"b"}, "b": {"c"}, "c": {"a"}}).static_order())
except graphlib.CycleError as err:
    cycle = err.args[1]
    print("CycleError is ValueError:", isinstance(err, ValueError))
    print("cycle starts and ends on the same node:", cycle[0] == cycle[-1], "| length:", len(cycle))
try:
    list(graphlib.TopologicalSorter({"x": {"x"}}).static_order())
except graphlib.CycleError:
    print("a node depending on itself is a cycle too")
# Gotcha: nodes must be hashable (strings, ints, tuples), like dict keys.

CycleError is ValueError: True
cycle starts and ends on the same node: True | length: 4
a node depending on itself is a cycle too


## 05. codecs - Encode And Decode

- str.encode("utf-8") and bytes.decode("utf-8") use codecs under the hood.
- The codecs module gives the lower-level API: codecs.encode/decode, lookup,
  and codecs that are not text<->bytes ("hex", "base64", "zlib" ...).
- codecs.lookup(name) returns CodecInfo; .name is the NORMALIZED name, so
  "UTF8", "utf_8", "U8" all become "utf-8".

In [7]:
text = "café ₹"
print("utf-8 bytes:", codecs.encode(text, "utf-8"))
print("utf-16-le bytes:", len(text.encode("utf-16-le")))
print("round trip:", codecs.decode(codecs.encode(text, "utf-8"), "utf-8") == text)
print("normalized names:", [codecs.lookup(n).name for n in ("UTF8", "utf_8", "latin1", "ASCII")])
print("bytes-to-bytes codec hex:", codecs.encode(b"\x00\xff", "hex"))
print("zlib round trip:", codecs.decode(codecs.encode(b"aaaa" * 10, "zlib"), "zlib") == b"aaaa" * 10)
try:
    "café".encode("ascii")
except UnicodeEncodeError as err:
    print("ascii cannot hold é:", err.reason)
print("a known-missing codec:", end=" ")
try:
    codecs.lookup("no-such-codec")
except LookupError:
    print("LookupError")

utf-8 bytes: b'caf\xc3\xa9 \xe2\x82\xb9'
utf-16-le bytes: 12
round trip: True
normalized names: ['utf-8', 'utf-8', 'iso8859-1', 'ascii']
bytes-to-bytes codec hex: b'00ff'
zlib round trip: True
ascii cannot hold é: ordinal not in range(128)
a known-missing codec: LookupError


## 06. Error Handlers

- Third argument "errors=" says what to do with characters that cannot be
  encoded / bytes that cannot be decoded.
  strict            raise (default)
  ignore            drop it silently (data loss!)
  replace           ? when encoding, U+FFFD when decoding
  backslashreplace  \xe9 style escapes (reversible with unicode_escape)
  xmlcharrefreplace &#233; (encoding only)
  namereplace       \N{LATIN SMALL LETTER E WITH ACUTE} (encoding only)
  surrogateescape   keep undecodable bytes so you can write them back later
- codecs.register_error(name, func) adds your own handler.

In [8]:
word = "café"
for handler in ("ignore", "replace", "backslashreplace", "xmlcharrefreplace", "namereplace"):
    print(f"{handler:18}", word.encode("ascii", errors=handler))
bad = b"caf\xe9"  # latin-1 bytes, not valid utf-8
print("decode replace:", bad.decode("utf-8", errors="replace"))
print("decode ignore:", bad.decode("utf-8", errors="ignore"))
kept = bad.decode("utf-8", errors="surrogateescape")
print("surrogateescape round trip:", kept.encode("utf-8", errors="surrogateescape") == bad)


def to_question_mark_pair(err):
    # Called for each bad span; returns (replacement text, position to resume)
    return ("<?>" * (err.end - err.start), err.end)


codecs.register_error("qmark3", to_question_mark_pair)
print("custom handler:", "a€b".encode("ascii", errors="qmark3"))

ignore             b'caf'
replace            b'caf?'
backslashreplace   b'caf\\xe9'
xmlcharrefreplace  b'caf&#233;'
namereplace        b'caf\\N{LATIN SMALL LETTER E WITH ACUTE}'
decode replace: caf�
decode ignore: caf
surrogateescape round trip: True
custom handler: b'a<?>b'


## 07. Incremental Decoders - Text Arriving In Chunks

- Network and file data comes in pieces. A multi-byte character (UTF-8 "é" =
  2 bytes, "₹" = 3) can be SPLIT across two chunks. Decoding each chunk alone
  fails or garbles it.
- codecs.getincrementaldecoder(enc)() remembers the unfinished bytes between
  calls. Pass final=True on the last chunk. getincrementalencoder is the
  encoder twin. codecs.iterdecode wraps this for an iterator of chunks.

In [9]:
data = "a₹b".encode("utf-8")
print("full bytes:", len(data))
pieces = [data[:2], data[2:]]  # cuts the rupee sign in half
try:
    "".join(p.decode("utf-8") for p in pieces)
except UnicodeDecodeError:
    print("chunk-by-chunk decode() fails on a split character")
dec = codecs.getincrementaldecoder("utf-8")()
out = []
for chunk in pieces:
    out.append(dec.decode(chunk))
out.append(dec.decode(b"", final=True))
print("incremental pieces:", out, "-> joined:", "".join(out))
print("iterdecode:", "".join(codecs.iterdecode(iter(pieces), "utf-8")))
enc = codecs.getincrementalencoder("utf-16")()
print("utf-16 encoder writes the BOM only once:", len(enc.encode("a")), len(enc.encode("b")))

full bytes: 5
chunk-by-chunk decode() fails on a split character
incremental pieces: ['a', '₹b', ''] -> joined: a₹b
iterdecode: a₹b
utf-16 encoder writes the BOM only once: 4 2


## 08. codecs.open vs open

- open(path, encoding="utf-8") is the modern answer: built in, fast, handles
  newlines in text mode, and is what everyone uses.
- codecs.open(path, mode, encoding) is the OLD Python 2 era tool. It always
  opens in BINARY underneath, does NOT translate line endings, and treats
  some unicode separators (like \x0b,  ) as line breaks.
- Use open(). Know codecs.open only so you can read older code.

In [10]:
tmp = tempfile.mkdtemp(prefix="codecs_demo_")
try:
    path = os.path.join(tmp, "t.txt")
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write("one two\nthree\n")
    with open(path, encoding="utf-8") as f:
        print("open() lines:", len(f.readlines()))
    with codecs.open(path, "r", encoding="utf-8") as f:
        print("codecs.open() lines (splits on U+2028):", len(f.readlines()))

    # 09. A Custom Codec And The BOM
    # ------------------------------------
    # - register a search function with codecs.register; it gets a name and
    #   returns a CodecInfo (or None if it is not yours).
    # - Here a toy "rot13-text" codec: str -> str, so we use the built-in
    #   rot_13 (which is exactly such a registered codec) and then add ours.
    # - BOM = Byte Order Mark, bytes at the start saying the byte order /
    #   encoding. "utf-8-sig" writes/removes the UTF-8 BOM (Excel likes it);
    #   plain "utf-16" writes a BOM and uses it when decoding.
    print("built-in rot13:", codecs.encode("Hello", "rot_13"))

    def make_shout_codec(name):
        if name != "shout":
            return None

        def encode(s, errors="strict"):
            return (s.upper().encode("utf-8", errors), len(s))

        def decode(b, errors="strict"):
            return (bytes(b).decode("utf-8", errors).lower(), len(b))

        return codecs.CodecInfo(encode, decode, name="shout")

    codecs.register(make_shout_codec)
    print("custom codec encode:", "quiet".encode("shout"))
    print("custom codec decode:", b"LOUD".decode("shout"))
    print("BOM constants:", codecs.BOM_UTF8, codecs.BOM_UTF16_LE)
    print("utf-8-sig writes BOM:", "hi".encode("utf-8-sig") == codecs.BOM_UTF8 + b"hi")
    print("utf-8 (plain) keeps BOM as a char:", repr(codecs.BOM_UTF8 + b"hi" == b"\xef\xbb\xbfhi" and (codecs.BOM_UTF8 + b"hi").decode("utf-8")))
    print("utf-8-sig removes BOM:", (codecs.BOM_UTF8 + b"hi").decode("utf-8-sig"))
    print("utf-16 starts with BOM:", "hi".encode("utf-16")[:2] in (codecs.BOM_UTF16_LE, codecs.BOM_UTF16_BE))
    print("utf-16-le never writes one:", "hi".encode("utf-16-le")[:2] == b"h\x00")
    # Gotcha: when CSV from Excel shows a strange "﻿" in the first column
    # header, read the file with encoding="utf-8-sig".

    # 10. getpass - Passwords And The Current User (comment-only prompts)
    # ------------------------------------
    # - getpass.getpass("Password: ") reads a line from the terminal WITHOUT
    #   echoing it. It is NOT run here, since it would wait for input.
    #       secret = getpass.getpass("Password: ")   # never echoed
    # - If there is no terminal (a pipe, CI) it warns and may read plain text.
    #   It does not encrypt or hide the value in memory; it only avoids the
    #   screen echo and shell history.
    # - getpass.getuser() reads LOGNAME / USER / LNAME / USERNAME, then the
    #   password database. Useful for log lines; never use it for security
    #   decisions because an environment variable can be faked.
    user = getpass.getuser()
    print("getuser returns a non-empty str:", isinstance(user, str) and len(user) > 0)
    print("getpass is a callable (not called):", callable(getpass.getpass))
    # Never put passwords on the command line or in source; use environment
    # variables, a secrets manager or the `keyring` package.

    # 11. mimetypes And colorsys
    # ------------------------------------
    # - mimetypes guesses a MIME type from a file NAME/URL (never opens the file).
    #   guess_type -> (type, encoding); guess_extension / guess_all_extensions
    #   go the other way. Used by web servers and email attachments.
    # - colorsys converts colours between RGB, HSV, HLS and YIQ. All values are
    #   floats from 0.0 to 1.0 (not 0-255).
    for name in ("a.html", "pic.png", "data.json", "archive.tar.gz", "noext"):
        print(f"  {name:15}", mimetypes.guess_type(name))
    print("extension for text/plain:", mimetypes.guess_extension("text/plain"))
    print("many extensions for image/jpeg include .jpg:", ".jpg" in mimetypes.guess_all_extensions("image/jpeg"))
    mimetypes.add_type("application/x-demo", ".demo")
    print("custom type registered:", mimetypes.guess_type("x.demo")[0])
    print("rgb red -> hsv:", colorsys.rgb_to_hsv(1, 0, 0))
    h, s, v = colorsys.rgb_to_hsv(0.0, 0.5, 1.0)
    print("blue-ish hue rounded:", round(h, 3), "| back to rgb:", tuple(round(c, 3) for c in colorsys.hsv_to_rgb(h, s, v)))
    r, g, b = (round(c * 255) for c in colorsys.hls_to_rgb(0.0, 0.5, 1.0))
    print("pure red via HLS, as 0-255:", (r, g, b))

    # 12. shlex - Splitting Like A Shell
    # ------------------------------------
    # - str.split() breaks on every space, so it ruins quoted arguments.
    #   shlex.split follows shell quoting rules: "double", 'single', backslash.
    # - shlex.quote makes ONE string safe to paste into a shell command.
    #   shlex.join(list) = quote each part, then join with spaces.
    # - Use shlex with subprocess to build or parse commands, but prefer a list
    #   of arguments with shell=False so no quoting is needed at all.
    line = 'run --name "My App" --tag=\'a b\' plain\\ space'
    print("str.split :", line.split())
    print("shlex.split:", shlex.split(line))
    print("quote safe word:", shlex.quote("safe-word_1.txt"))
    print("quote risky:", shlex.quote("a b; rm -rf x"))
    print("join:", shlex.join(["echo", "it's", "two words"]))
    print("comments skipped with comments=True:", shlex.split("a b # note", comments=True))
    try:
        shlex.split('unterminated "quote')
    except ValueError as err:
        print("bad quoting raises ValueError:", err)

    # 13. fileinput - Loop Over Many Files Like One
    # ------------------------------------
    # - fileinput.input(files) reads lines from several files as one stream;
    #   with no files it reads sys.argv[1:] or stdin (like Unix `cat`, `sed`).
    # - fileinput.filename(), lineno() (running total), filelineno() (per file).
    # - inplace=True redirects print() INTO the file being read, so you can
    #   edit files in place (backup="" or ".bak" keeps the original).
    a = os.path.join(tmp, "a.txt")
    b = os.path.join(tmp, "b.txt")
    with open(a, "w") as f:
        f.write("alpha\nbeta\n")
    with open(b, "w") as f:
        f.write("gamma\n")
    with fileinput.input(files=(a, b)) as fi:
        for line in fi:
            print(f"  {os.path.basename(fi.filename())} total={fi.lineno()} per-file={fi.filelineno()}: {line.rstrip()}")
    with fileinput.input(files=(a,), inplace=True) as fi:
        for line in fi:
            print(line.rstrip().upper())  # goes into the FILE, not the screen
    with open(a) as f:
        print("after in-place edit:", f.read().split())
    # Gotcha: only one fileinput.input() may be active at a time (module-level
    # state); use the FileInput class for independent readers.

    # 14. tokenize And keyword
    # ------------------------------------
    # - keyword.kwlist = all reserved words; iskeyword("x") tests one;
    #   softkwlist = "soft" keywords (match, case, _, type) that are only
    #   special in certain positions and may still be variable names.
    # - tokenize splits SOURCE into tokens (NAME, NUMBER, STRING, OP, COMMENT,
    #   NEWLINE...) keeping comments - which ast throws away.
    # - Use it for syntax highlighters and formatters.
    print("is 'class' a keyword:", keyword.iskeyword("class"), "| 'print':", keyword.iskeyword("print"))
    print("count of keywords is more than 30:", len(keyword.kwlist) > 30)
    print("soft keywords include match:", "match" in keyword.softkwlist)
    src = "total = price * 2  # double\n"
    toks = list(tokenize.generate_tokens(io.StringIO(src).readline))
    print([(tokenize.tok_name[t.type], t.string) for t in toks if t.type not in (tokenize.NEWLINE, tokenize.ENDMARKER)])
    print("comment survives tokenizing:", any(t.type == tokenize.COMMENT for t in toks))
    # tokenize.open(path) opens a Python file using its declared encoding.

    # 15. zipapp - A Whole Program In One File
    # ------------------------------------
    # - A .pyz file is a zip with a __main__.py that `python app.pyz` runs.
    #   zipapp.create_archive(source_dir, target, interpreter=..., main=...)
    #   builds one. Good for shipping a script plus its pure-Python helpers.
    # - Limits: only pure Python (C extensions cannot load from inside a zip).
    # - Command line form:  python -m zipapp app_dir -o app.pyz -p "/usr/bin/env python3"
    app_dir = os.path.join(tmp, "myapp")
    os.mkdir(app_dir)
    with open(os.path.join(app_dir, "__main__.py"), "w") as f:
        f.write("print('hello from inside a zipapp')\n")
    target = os.path.join(tmp, "myapp.pyz")
    zipapp.create_archive(app_dir, target)
    print("archive created:", os.path.exists(target), "| is zipapp file:", zipapp.get_interpreter(target) is None)
    import runpy
    runpy.run_path(target, run_name="__main__")  # same effect as `python myapp.pyz`

    # 16. pkgutil And importlib.resources
    # ------------------------------------
    # - pkgutil.iter_modules(path) lists importable modules WITHOUT importing
    #   them; walk_packages recurses; pkgutil.get_data(pkg, "file") reads a
    #   data file shipped inside a package.
    # - importlib.resources is the modern way to read package data files:
    #   files("pkg") returns a Traversable (like a Path that also works inside
    #   zips). Do not build paths from __file__; that breaks in zipped installs.
    # - importlib.import_module(name_string) imports by name at run time (plugins).
    names = {m.name for m in pkgutil.iter_modules(["."]) if m.name.startswith("zz_")}
    print("iter_modules on a path with no 'zz_' modules:", names)
    pkg_dir = os.path.join(tmp, "demo_pkg")
    os.mkdir(pkg_dir)
    for fname, body in (("__init__.py", ""), ("hello.txt", "hi from package data\n")):
        with open(os.path.join(pkg_dir, fname), "w") as f:
            f.write(body)
    print("pkgutil sees module:", [m.name for m in pkgutil.iter_modules([tmp]) if m.name == "demo_pkg"])
    sys.path.insert(0, tmp)
    try:
        importlib.invalidate_caches()
        pkg = importlib.import_module("demo_pkg")
        print("importlib.resources read_text:", importlib.resources.files("demo_pkg").joinpath("hello.txt").read_text().strip())
        print("pkgutil.get_data:", pkgutil.get_data("demo_pkg", "hello.txt"))
        print("is_resource-style check via files():", importlib.resources.files("demo_pkg").joinpath("hello.txt").is_file())
    finally:
        sys.path.remove(tmp)
        sys.modules.pop("demo_pkg", None)
        sys.modules.pop("demo_pkg.__pycache__", None)
    print("os module loaded by name:", importlib.import_module("os") is os)
finally:
    import shutil
    shutil.rmtree(tmp, ignore_errors=True)
print("temp dir removed:", not os.path.exists(tmp))

open() lines: 2
codecs.open() lines (splits on U+2028): 3
built-in rot13: Uryyb
custom codec encode: b'QUIET'
custom codec decode: loud
BOM constants: b'\xef\xbb\xbf' b'\xff\xfe'
utf-8-sig writes BOM: True
utf-8 (plain) keeps BOM as a char: '\ufeffhi'
utf-8-sig removes BOM: hi
utf-16 starts with BOM: True
utf-16-le never writes one: True
getuser returns a non-empty str: True
getpass is a callable (not called): True
  a.html          ('text/html', None)
  pic.png         ('image/png', None)
  data.json       ('application/json', None)
  archive.tar.gz  ('application/x-tar', 'gzip')
  noext           (None, None)
extension for text/plain: .txt
many extensions for image/jpeg include .jpg: True
custom type registered: application/x-demo
rgb red -> hsv: (0.0, 1.0, 1)
blue-ish hue rounded: 0.583 | back to rgb: (0.0, 0.5, 1.0)
pure red via HLS, as 0-255: (255, 0, 0)
str.split : ['run', '--name', '"My', 'App"', "--tag='a", "b'", 'plain\\', 'space']
shlex.split: ['run', '--name', 'My App', '--t

## 17. platform And sysconfig - Quick Tour

- platform answers "where am I running?": system(), machine(), release(),
  python_version(), python_implementation(), node() (host name).
- sys.platform is a short string ("linux", "darwin", "win32") - best for
  quick if-checks. os.name is "posix" or "nt".
- sysconfig describes how Python was built: get_paths() (install dirs),
  get_config_var("EXT_SUFFIX"), get_platform(). Used by build tools.
- Their exact values differ per machine, so we print only checks about them.

In [11]:
print("platform.system is one of the usual names:", platform.system() in ("Linux", "Darwin", "Windows", "FreeBSD", "OpenBSD", "Java", ""))
print("python_implementation is CPython here:", platform.python_implementation() == "CPython")
print("python_version has 3 parts:", len(platform.python_version().split(".")) == 3)
print("machine non-empty:", bool(platform.machine()))
print("sys.platform is str:", isinstance(sys.platform, str), "| os.name in (posix, nt):", os.name in ("posix", "nt"))
paths = sysconfig.get_paths()
print("sysconfig has stdlib and purelib keys:", {"stdlib", "purelib"} <= set(paths))
print("stdlib dir exists:", os.path.isdir(paths["stdlib"]))
print("EXT_SUFFIX ends with .so/.pyd:", str(sysconfig.get_config_var("EXT_SUFFIX")).endswith((".so", ".pyd", ".dylib")))
print("platform.uname fields:", sorted(platform.uname()._fields))
# Gotcha: platform.platform()/node() reveal machine details; never print them
# into logs you share. And do not parse them; compare to known constants.

platform.system is one of the usual names: True
python_implementation is CPython here: True
python_version has 3 parts: True
machine non-empty: True
sys.platform is str: True | os.name in (posix, nt): True
sysconfig has stdlib and purelib keys: True
stdlib dir exists: True
EXT_SUFFIX ends with .so/.pyd: True
platform.uname fields: ['machine', 'node', 'processor', 'release', 'system', 'version']


## 18. site And sys.path Recap

- sys.path = the ordered list of places `import` searches. Order matters:
  the script's own folder first, then PYTHONPATH, then the standard library,
  then site-packages (third-party installs).
- The `site` module runs at startup, adds site-packages (and a per-user one,
  plus any .pth file entries), and sets sys.prefix / base_prefix.
- Virtual environment: sys.prefix != sys.base_prefix.
- Gotcha: a file named like a stdlib module (random.py, json.py) in your
  script folder SHADOWS the real one, and the error looks bizarre.

In [12]:
print("sys.path is a list of strings:", isinstance(sys.path, list) and all(isinstance(p, str) for p in sys.path))
print("script directory is searched first:", os.path.abspath(sys.path[0]) == os.path.dirname(os.path.abspath(__file__)))
print("site.getsitepackages exists:", callable(site.getsitepackages))
print("user site flag is bool:", isinstance(site.ENABLE_USER_SITE, (bool, type(None))))
print("venv check is a bool (value depends on how you run this):", isinstance(sys.prefix != sys.base_prefix, bool))
print("os module comes from the stdlib dir:", os.path.dirname(os.__file__) == sysconfig.get_paths()["stdlib"])
print("sys.modules caches imports, 'sys' in it:", "sys" in sys.modules)

sys.path is a list of strings: True
script directory is searched first: True
site.getsitepackages exists: True
user site flag is bool: True
venv check is a bool (value depends on how you run this): True
os module comes from the stdlib dir: True
sys.modules caches imports, 'sys' in it: True


## 19. webbrowser And smtplib - Comment Only (side effects!)

- webbrowser.open("https://example.com") opens the user's default browser.
  open_new_tab(url) too. It is NOT called here: it would pop a window, and
  returns True/False without proving a page loaded. A `python -m webbrowser URL`
  command line also exists.
- smtplib sends email by talking to an SMTP server:
      import smtplib
      from email.message import EmailMessage
      msg = EmailMessage(); msg["From"] = ...; msg["To"] = ...
      msg["Subject"] = "Hi"; msg.set_content("Body text")
      with smtplib.SMTP("smtp.example.com", 587) as s:
          s.starttls()            # upgrade to encrypted before logging in
          s.login(user, password) # from env vars or a secrets store
          s.send_message(msg)
  or SMTP_SSL(host, 465) for implicit TLS.
  NOT run: it needs a real server and credentials, and it would send mail.
- For tests, use a local debugging server or mock smtplib.SMTP; never
  test against a real inbox. Build messages with email.message, not by
  gluing header strings (header injection via newlines in user input).

In [13]:
print("webbrowser and smtplib are importable but intentionally never called here")
print("EmailMessage can be built offline without sending:", end=" ")
from email.message import EmailMessage
em = EmailMessage()
em["Subject"] = "Hi"
em.set_content("Body")
print(em["Subject"], "|", em.get_content().strip())

webbrowser and smtplib are importable but intentionally never called here
EmailMessage can be built offline without sending: Hi | Body


## 20. tkinter And turtle - GUI Modules (explained, not run)

- tkinter: the standard GUI toolkit (windows, buttons, labels, canvas),
  a wrapper around Tcl/Tk. A program builds widgets and then calls
  root.mainloop(), which BLOCKS and waits for clicks (the event loop).
      import tkinter as tk
      root = tk.Tk()
      tk.Label(root, text="Hello").pack()
      tk.Button(root, text="Quit", command=root.destroy).pack()
      root.mainloop()
- turtle: a teaching module built on tkinter. You move a "turtle" on screen:
      import turtle
      t = turtle.Turtle(); t.forward(100); t.left(90); turtle.done()
- Both need a display, so they fail on servers/CI (TclError: no display),
  and some Python builds ship without Tk. They open a window and block, so
  they are not run in this file.
- Tips: keep long work out of callbacks (it freezes the window); use
  root.after(ms, fn) for timers; only touch widgets from the main thread.
- For real apps people often choose web UIs, Qt (PySide), or Kivy; tkinter
  is great for small tools and learning event-driven programming.

In [14]:
print("tkinter importable (checked without opening a window):", importlib.util.find_spec("tkinter") is not None)

tkinter importable (checked without opening a window): True


## 21. What To Remember

- graphlib: dependencies -> order. static_order() for simple, get_ready/done
  for parallel waves; CycleError for loops.
- codecs: pick errors= on purpose; incremental decoders for chunked bytes;
  open() not codecs.open(); utf-8-sig for BOM-prefixed files.
- small helpers: shlex (quoting), mimetypes (types by name), colorsys (0-1
  colours), fileinput (many files as one), tokenize/keyword (source tools).
- environment: platform/sysconfig describe the machine; sys.path/site decide
  what imports. GUI/web/mail modules cause side effects: guard them.

In [15]:
print("done:", True)

done: True
